# Advanced `JSONDecoder` Tutorial — Problems, Reasoning, and Solutions

In this notebook we are going to continue working with Python's `json` module,
but this time we will focus specifically on **advanced decoding problems**.

The style of this notebook is intentionally tutorial-oriented.

Instead of jumping directly from a problem statement to a finished solution,
we will usually:

1. understand the problem,
2. inspect what Python's standard decoder gives us,
3. identify the best extension point,
4. build a small part of the solution,
5. test that part,
6. improve it,
7. combine the pieces,
8. test edge cases.

The main topic is still custom JSON decoding, especially:

- `json.JSONDecoder`
- subclassing `JSONDecoder`
- `object_hook`
- `object_pairs_hook`
- `parse_float`
- `parse_int`
- `parse_constant`
- `raw_decode`
- validation after decoding
- tagged JSON objects
- nested object reconstruction
- defensive decoding


A key theme throughout this notebook is this:

> **Do not override `JSONDecoder.decode` unless the customization really needs
> access to the complete decoded document or the complete JSON string.**

For individual JSON objects nested anywhere inside a document, `object_hook`
is often a better tool.

For duplicate-key detection, `object_pairs_hook` is usually the correct tool.

For numeric conversion, `parse_float` and `parse_int` are usually cleaner than
post-processing the decoded result.


In [2]:
import json

from dataclasses import dataclass
from datetime import datetime, timezone
from decimal import Decimal
from pprint import pprint
from uuid import UUID


# Problem 1 — Reconstructing a Custom `Point` Object

Suppose we receive JSON such as:

```json
{
    "name": "line",
    "start": {
        "_type": "point",
        "x": 10,
        "y": 20
    },
    "end": {
        "_type": "point",
        "x": 30,
        "y": 40
    }
}
```

We would like the two dictionaries representing points to become Python
`Point` objects automatically.

Let's build this gradually.


## Step 1 — Define the Python class

We first need a Python type to reconstruct.

Using a dataclass makes the example compact and gives us a useful
representation and equality behavior automatically.


In [3]:
@dataclass(frozen=True)
class Point:
    x: object
    y: object


## Step 2 — See what ordinary decoding produces

Before customizing anything, always inspect the default behavior.

That tells us exactly what part of the decoded object needs transformation.


In [4]:
j = '''
{
    "name": "line",
    "start": {
        "_type": "point",
        "x": 10,
        "y": 20
    },
    "end": {
        "_type": "point",
        "x": 30,
        "y": 40
    }
}
'''

result = json.loads(j)
pprint(result)


{'end': {'_type': 'point', 'x': 30, 'y': 40},
 'name': 'line',
 'start': {'_type': 'point', 'x': 10, 'y': 20}}


As expected, the JSON objects become ordinary dictionaries.

One possible approach would be:

1. decode the whole document,
2. recursively search every dictionary and list,
3. replace point dictionaries afterward.

That works, but the `json` module already provides a better mechanism:
`object_hook`.


## Step 3 — Observe how `object_hook` is called

`object_hook` receives each decoded JSON object.

Let's temporarily write a hook that only prints what it receives.


In [5]:
def debug_hook(obj):
    print("HOOK RECEIVED:", obj)
    return obj

json.loads(j, object_hook=debug_hook)


HOOK RECEIVED: {'_type': 'point', 'x': 10, 'y': 20}
HOOK RECEIVED: {'_type': 'point', 'x': 30, 'y': 40}
HOOK RECEIVED: {'name': 'line', 'start': {'_type': 'point', 'x': 10, 'y': 20}, 'end': {'_type': 'point', 'x': 30, 'y': 40}}


{'name': 'line',
 'start': {'_type': 'point', 'x': 10, 'y': 20},
 'end': {'_type': 'point', 'x': 30, 'y': 40}}

Notice something important:

The inner objects are processed before their containing object.

That means when the top-level dictionary is passed to the hook, the decoder has
already had an opportunity to transform the nested dictionaries.

This bottom-up behavior is exactly what we need.


## Step 4 — Convert only dictionaries tagged as points

We will use the `_type` key as an explicit marker.

If `_type` is not `"point"`, the hook simply returns the dictionary unchanged.


In [6]:
def point_hook(obj):
    if obj.get("_type") == "point":
        return Point(obj["x"], obj["y"])

    return obj


In [7]:
result = json.loads(j, object_hook=point_hook)
pprint(result)

print(type(result["start"]))
print(type(result["end"]))


{'end': Point(x=30, y=40), 'name': 'line', 'start': Point(x=10, y=20)}
<class '__main__.Point'>
<class '__main__.Point'>


## Step 5 — Improve the solution with validation

The previous version trusts that every object tagged as `"point"` has exactly
the structure we expect.

That may be too optimistic.

For a protocol-style format, it is usually better to reject malformed tagged
objects immediately.


In [8]:
def point_hook(obj):
    if obj.get("_type") != "point":
        return obj

    expected = {"_type", "x", "y"}

    if set(obj) != expected:
        raise ValueError(
            f"Malformed point. Expected keys {expected}, got {set(obj)}"
        )

    return Point(obj["x"], obj["y"])


In [9]:
result = json.loads(j, object_hook=point_hook)
pprint(result)

assert result["start"] == Point(10, 20)
assert result["end"] == Point(30, 40)


{'end': Point(x=30, y=40), 'name': 'line', 'start': Point(x=10, y=20)}


## Problem 1 — Final lesson

For tagged objects nested anywhere in a JSON document:

- `object_hook` is usually simpler than manually traversing the decoded result,
- nested objects are transformed before their parents,
- validation can be performed at the moment a tagged object is reconstructed.


# Problem 2 — Decode Floating-Point Numbers as `Decimal`

Suppose we are decoding financial data:

```json
{
    "unit_price": 0.1,
    "quantity": 3,
    "tax_rate": 0.075
}
```

For financial calculations, binary floating-point values are often undesirable.

We would like JSON floating-point literals to become `Decimal` objects.


## Step 1 — Inspect the ordinary result


In [10]:
financial_json = '''
{
    "unit_price": 0.1,
    "quantity": 3,
    "tax_rate": 0.075
}
'''

ordinary = json.loads(financial_json)

print(ordinary)
print(type(ordinary["unit_price"]))
print(type(ordinary["tax_rate"]))


{'unit_price': 0.1, 'quantity': 3, 'tax_rate': 0.075}
<class 'float'>
<class 'float'>


The decoder converted JSON numbers containing decimal points into Python
`float` objects.

We *could* recursively walk the result and replace floats with `Decimal`, but
that would already be too late: the conversion to binary floating point has
already happened.

Instead, the decoder provides `parse_float`.


## Step 2 — Use `parse_float`

The function supplied to `parse_float` receives the original number token as a
string.

That makes `Decimal` an excellent fit.


In [11]:
decimal_result = json.loads(
    financial_json,
    parse_float=Decimal
)

print(decimal_result)
print(type(decimal_result["unit_price"]))
print(type(decimal_result["tax_rate"]))


{'unit_price': Decimal('0.1'), 'quantity': 3, 'tax_rate': Decimal('0.075')}
<class 'decimal.Decimal'>
<class 'decimal.Decimal'>


## Step 3 — Perform exact decimal arithmetic


In [12]:
unit_price = decimal_result["unit_price"]
quantity = decimal_result["quantity"]
tax_rate = decimal_result["tax_rate"]

subtotal = unit_price * quantity
tax = subtotal * tax_rate
total = subtotal + tax

print("subtotal:", subtotal)
print("tax:", tax)
print("total:", total)


subtotal: 0.3
tax: 0.0225
total: 0.3225


## Step 4 — Compare `Decimal("0.1")` with `Decimal(0.1)`

This distinction is important.

`Decimal("0.1")` reads the intended decimal representation directly.

`Decimal(0.1)` starts from an already-created binary float.


In [13]:
print("Decimal from string:", Decimal("0.1"))
print("Decimal from float :", Decimal(0.1))


Decimal from string: 0.1
Decimal from float : 0.1000000000000000055511151231257827021181583404541015625


## Problem 2 — Final lesson

If numeric precision matters, perform the conversion **during parsing** whenever
possible.

For JSON floating-point literals, `parse_float=Decimal` is usually much better
than decoding to floats first and fixing them later.


# Problem 3 — Build a Decoder that Handles Both `Decimal` and `Point`

Now let's combine the previous two ideas.

Requirements:

- all JSON floating-point literals should become `Decimal`,
- tagged point objects should become `Point`,
- ordinary dictionaries should stay dictionaries.

We could specify `parse_float=Decimal` and `object_hook=point_hook` every time we
call `json.loads`.

But suppose this is a standard decoding policy for our application.

It may be cleaner to bundle that policy into a decoder class.


## Step 1 — A tempting but clumsy approach

We could override `decode` and then call another decoder from inside it.

For example:

```python
class MyDecoder(json.JSONDecoder):
    def decode(self, s):
        return json.loads(
            s,
            parse_float=Decimal,
            object_hook=point_hook
        )
```

This works in simple cases, but it is not the cleanest design.

We are creating a decoder subclass only to call a completely separate decoding
operation.

A better approach is to configure the parent decoder.


## Step 2 — Configure `JSONDecoder` through `__init__`


In [14]:
class PointDecimalDecoder(json.JSONDecoder):
    def __init__(self, *args, **kwargs):
        kwargs.setdefault("parse_float", Decimal)
        kwargs.setdefault("object_hook", point_hook)

        super().__init__(*args, **kwargs)


## Step 3 — Test the custom decoder


In [15]:
j = '''
{
    "name": "measurement",
    "scale": 0.125,
    "position": {
        "_type": "point",
        "x": 10.5,
        "y": -3.75
    }
}
'''

result = json.loads(j, cls=PointDecimalDecoder)
pprint(result)

print(type(result["scale"]))
print(type(result["position"]))
print(type(result["position"].x))
print(type(result["position"].y))


{'name': 'measurement',
 'position': Point(x=Decimal('10.5'), y=Decimal('-3.75')),
 'scale': Decimal('0.125')}
<class 'decimal.Decimal'>
<class '__main__.Point'>
<class 'decimal.Decimal'>
<class 'decimal.Decimal'>


The decoder now bundles both behaviors.

The point's coordinates also became `Decimal` values because numeric parsing
happened before `object_hook` reconstructed the `Point`.


## Problem 3 — Final solution check


In [16]:
assert result["scale"] == Decimal("0.125")
assert result["position"] == Point(
    Decimal("10.5"),
    Decimal("-3.75")
)


# Problem 4 — Reconstruct Several Custom Types

Suppose our JSON protocol supports more than one tagged type:

- point,
- set,
- tuple,
- complex number.

We want to avoid writing a huge chain such as:

```python
if tag == "point":
    ...
elif tag == "set":
    ...
elif tag == "tuple":
    ...
elif tag == "complex":
    ...
```

Instead, we will build a small decoder registry.


## Step 1 — Write one function per type

Keeping each conversion separate makes the logic easier to test.


In [17]:
def decode_point(obj):
    if set(obj) != {"_type", "x", "y"}:
        raise ValueError("Malformed point")

    return Point(obj["x"], obj["y"])


def decode_set(obj):
    if set(obj) != {"_type", "items"}:
        raise ValueError("Malformed set")

    return set(obj["items"])


def decode_tuple(obj):
    if set(obj) != {"_type", "items"}:
        raise ValueError("Malformed tuple")

    return tuple(obj["items"])


def decode_complex(obj):
    if set(obj) != {"_type", "real", "imag"}:
        raise ValueError("Malformed complex number")

    return complex(obj["real"], obj["imag"])


## Step 2 — Create the registry

The registry maps a tag name to the function responsible for decoding it.


In [18]:
TYPE_DECODERS = {
    "point": decode_point,
    "set": decode_set,
    "tuple": decode_tuple,
    "complex": decode_complex,
}


## Step 3 — Write the generic hook

The hook now has only three responsibilities:

1. determine whether the object is tagged,
2. find the corresponding decoder,
3. call it.


In [19]:
def registry_hook(obj):
    tag = obj.get("_type")

    if tag is None:
        return obj

    if tag not in TYPE_DECODERS:
        raise ValueError(f"Unknown tagged type: {tag!r}")

    decoder = TYPE_DECODERS[tag]
    return decoder(obj)


## Step 4 — Test nested values

The tuple below contains two tagged point objects.

Because `object_hook` works from the inside outward, the point dictionaries are
already converted before the tuple decoder is called.


In [20]:
j = '''
{
    "origin": {
        "_type": "point",
        "x": 0,
        "y": 0
    },

    "labels": {
        "_type": "set",
        "items": ["red", "green", "blue"]
    },

    "segment": {
        "_type": "tuple",
        "items": [
            {"_type": "point", "x": 1, "y": 2},
            {"_type": "point", "x": 3, "y": 4}
        ]
    },

    "z": {
        "_type": "complex",
        "real": 2,
        "imag": -5
    }
}
'''

result = json.loads(j, object_hook=registry_hook)
pprint(result)


{'labels': {'blue', 'green', 'red'},
 'origin': Point(x=0, y=0),
 'segment': (Point(x=1, y=2), Point(x=3, y=4)),
 'z': (2-5j)}


In [21]:
assert result["origin"] == Point(0, 0)
assert result["labels"] == {"red", "green", "blue"}
assert result["segment"] == (Point(1, 2), Point(3, 4))
assert result["z"] == complex(2, -5)


## Problem 4 — Final lesson

A registry is useful because:

- each custom type has isolated decoding logic,
- adding a new type does not require rewriting the main hook,
- unknown type tags can be rejected centrally,
- nested tagged values continue to work automatically.


# Problem 5 — Reject Duplicate JSON Keys

Consider this JSON:

```json
{
    "username": "alice",
    "role": "user",
    "role": "admin"
}
```

What should happen?

Let's first see what Python does by default.


In [22]:
duplicate_json = '''
{
    "username": "alice",
    "role": "user",
    "role": "admin"
}
'''

print(json.loads(duplicate_json))


{'username': 'alice', 'role': 'admin'}


The first `"role"` value disappeared.

By the time an ordinary dictionary exists, the duplicate information is gone.

That means `object_hook` cannot detect this problem.

We need an earlier extension point: `object_pairs_hook`.


## Step 1 — See the key/value pairs before dictionary construction


In [23]:
def debug_pairs_hook(pairs):
    print("PAIRS:", pairs)
    return dict(pairs)

json.loads(duplicate_json, object_pairs_hook=debug_pairs_hook)


PAIRS: [('username', 'alice'), ('role', 'user'), ('role', 'admin')]


{'username': 'alice', 'role': 'admin'}

The duplicate is visible in the list of pairs.

Now we can reject it.


## Step 2 — Build a duplicate-checking function


In [24]:
def no_duplicates(pairs):
    result = {}

    for key, value in pairs:
        if key in result:
            raise ValueError(
                f"Duplicate JSON key detected: {key!r}"
            )

        result[key] = value

    return result


## Step 3 — Test valid JSON


In [25]:
valid_json = '''
{
    "username": "alice",
    "role": "admin"
}
'''

print(json.loads(
    valid_json,
    object_pairs_hook=no_duplicates
))


{'username': 'alice', 'role': 'admin'}


## Step 4 — Test invalid JSON


In [26]:
try:
    json.loads(
        duplicate_json,
        object_pairs_hook=no_duplicates
    )
except ValueError as exc:
    print("Rejected:", exc)


Rejected: Duplicate JSON key detected: 'role'


## Important detail — `object_pairs_hook` and `object_hook`

If both are supplied, `object_pairs_hook` takes priority.

So if we need both:

- duplicate-key detection,
- tagged object conversion,

we should usually combine those behaviors in the pairs hook.


# Problem 6 — Duplicate-Key Detection + Tagged Objects

Let's combine the previous problem with point reconstruction.

Requirements:

- duplicate keys must be rejected,
- point objects must still become `Point`,
- ordinary objects must become dictionaries.


## Step 1 — Convert the pair list to a dictionary safely


In [27]:
def pairs_to_unique_dict(pairs):
    obj = {}

    for key, value in pairs:
        if key in obj:
            raise ValueError(f"Duplicate key: {key!r}")

        obj[key] = value

    return obj


## Step 2 — Apply tagged conversion after duplicate checking


In [28]:
def typed_pairs_hook(pairs):
    obj = pairs_to_unique_dict(pairs)

    if obj.get("_type") == "point":
        return decode_point(obj)

    return obj


## Step 3 — Test it


In [29]:
j = '''
{
    "name": "rectangle",
    "top_left": {
        "_type": "point",
        "x": -1.5,
        "y": 2.5
    },
    "bottom_right": {
        "_type": "point",
        "x": 5.25,
        "y": -3.75
    }
}
'''

result = json.loads(
    j,
    object_pairs_hook=typed_pairs_hook,
    parse_float=Decimal
)

pprint(result)


{'bottom_right': Point(x=Decimal('5.25'), y=Decimal('-3.75')),
 'name': 'rectangle',
 'top_left': Point(x=Decimal('-1.5'), y=Decimal('2.5'))}


In [30]:
assert result["top_left"] == Point(
    Decimal("-1.5"),
    Decimal("2.5")
)

assert result["bottom_right"] == Point(
    Decimal("5.25"),
    Decimal("-3.75")
)


# Problem 7 — Strictly Reject `NaN` and Infinity

Python's JSON decoder accepts these special tokens:

```text
NaN
Infinity
-Infinity
```

They are not part of strict JSON.

Let's see the default behavior.


In [31]:
examples = [
    '{"value": NaN}',
    '{"value": Infinity}',
    '{"value": -Infinity}',
]

for text in examples:
    result = json.loads(text)
    print(text, "->", result)


{"value": NaN} -> {'value': nan}
{"value": Infinity} -> {'value': inf}
{"value": -Infinity} -> {'value': -inf}


If our format requires strict JSON, we should reject these tokens.

The correct extension point is `parse_constant`.


## Step 1 — Write a rejecting function


In [32]:
def reject_constant(token):
    raise ValueError(
        f"Non-standard numeric constant is not allowed: {token}"
    )


## Step 2 — Use it with `json.loads`


In [33]:
for text in examples:
    try:
        json.loads(
            text,
            parse_constant=reject_constant
        )
    except ValueError as exc:
        print("Rejected:", exc)


Rejected: Non-standard numeric constant is not allowed: NaN
Rejected: Non-standard numeric constant is not allowed: Infinity
Rejected: Non-standard numeric constant is not allowed: -Infinity


## Step 3 — Verify ordinary numbers still work


In [34]:
print(json.loads(
    '{"a": 1, "b": 2.5}',
    parse_constant=reject_constant,
    parse_float=Decimal
))


{'a': 1, 'b': Decimal('2.5')}


# Problem 8 — Bound Integer Values During Parsing

Imagine that our application accepts integers only in this range:

```text
-1,000,000 <= n <= 1,000,000
```

We want out-of-range values to fail during decoding.

Again, the decoder already has the correct hook for this: `parse_int`.


## Step 1 — Write the integer parser

The parser receives the original JSON integer token as a string.


In [35]:
MIN_INT = -1_000_000
MAX_INT = 1_000_000


def bounded_int(token):
    value = int(token)

    if not MIN_INT <= value <= MAX_INT:
        raise ValueError(
            f"Integer {value} is outside "
            f"[{MIN_INT}, {MAX_INT}]"
        )

    return value


## Step 2 — Test values inside the allowed range


In [36]:
print(json.loads(
    '{"a": 10, "b": -999999}',
    parse_int=bounded_int
))


{'a': 10, 'b': -999999}


## Step 3 — Test an oversized value


In [37]:
try:
    json.loads(
        '{"value": 999999999999999}',
        parse_int=bounded_int
    )
except ValueError as exc:
    print("Rejected:", exc)


Rejected: Integer 999999999999999 is outside [-1000000, 1000000]


## Step 4 — Build a stricter numeric decoder

Now we combine:

- bounded integers,
- `Decimal` floating-point values,
- rejection of `NaN` and infinities.


In [38]:
class StrictNumberDecoder(json.JSONDecoder):
    def __init__(self, *args, **kwargs):
        kwargs.setdefault("parse_int", bounded_int)
        kwargs.setdefault("parse_float", Decimal)
        kwargs.setdefault("parse_constant", reject_constant)

        super().__init__(*args, **kwargs)


In [39]:
result = json.loads(
    '{"count": 10, "price": 12.75}',
    cls=StrictNumberDecoder
)

print(result)
print(type(result["count"]))
print(type(result["price"]))


{'count': 10, 'price': Decimal('12.75')}
<class 'int'>
<class 'decimal.Decimal'>


# Problem 9 — Decode `datetime` and `UUID`

Many real-world JSON APIs represent richer values as strings.

For example:

```json
{
    "_type": "datetime",
    "value": "2026-08-07T12:30:00+00:00"
}
```

and:

```json
{
    "_type": "uuid",
    "value": "550e8400-e29b-41d4-a716-446655440000"
}
```

We will add both to our tagged-object system.


## Step 1 — Decode a datetime

We will require timezone information.

This prevents accidental mixing of timezone-aware and timezone-naive values.


In [40]:
def decode_datetime(obj):
    if set(obj) != {"_type", "value"}:
        raise ValueError("Malformed datetime object")

    try:
        value = datetime.fromisoformat(obj["value"])
    except (TypeError, ValueError) as exc:
        raise ValueError(
            f"Invalid datetime value: {obj['value']!r}"
        ) from exc

    if value.tzinfo is None:
        raise ValueError(
            "Datetime values must include timezone information"
        )

    return value


## Step 2 — Decode a UUID


In [41]:
def decode_uuid(obj):
    if set(obj) != {"_type", "value"}:
        raise ValueError("Malformed UUID object")

    try:
        return UUID(obj["value"])
    except (TypeError, ValueError, AttributeError) as exc:
        raise ValueError(
            f"Invalid UUID value: {obj['value']!r}"
        ) from exc


## Step 3 — Extend the registry


In [42]:
EXTENDED_TYPE_DECODERS = {
    **TYPE_DECODERS,
    "datetime": decode_datetime,
    "uuid": decode_uuid,
}


def extended_registry_hook(obj):
    tag = obj.get("_type")

    if tag is None:
        return obj

    if tag not in EXTENDED_TYPE_DECODERS:
        raise ValueError(f"Unknown type tag: {tag!r}")

    return EXTENDED_TYPE_DECODERS[tag](obj)


## Step 4 — Test a nested document


In [43]:
j = '''
{
    "id": {
        "_type": "uuid",
        "value": "550e8400-e29b-41d4-a716-446655440000"
    },

    "created_at": {
        "_type": "datetime",
        "value": "2026-08-07T12:30:00+00:00"
    },

    "location": {
        "_type": "point",
        "x": 23.5,
        "y": 42.25
    }
}
'''

result = json.loads(
    j,
    object_hook=extended_registry_hook,
    parse_float=Decimal
)

pprint(result)


{'created_at': datetime.datetime(2026, 8, 7, 12, 30, tzinfo=datetime.timezone.utc),
 'id': UUID('550e8400-e29b-41d4-a716-446655440000'),
 'location': Point(x=Decimal('23.5'), y=Decimal('42.25'))}


In [44]:
assert isinstance(result["id"], UUID)
assert isinstance(result["created_at"], datetime)
assert result["created_at"].tzinfo is not None
assert result["location"] == Point(
    Decimal("23.5"),
    Decimal("42.25")
)


# Problem 10 — Decode Multiple JSON Values from One String

Consider this text:

```text
{"id": 1}
{"id": 2}
[10, 20, 30]
"finished"
```

As a whole, this is not one valid JSON document.

But it contains several valid JSON values next to each other.

The normal `json.loads` function expects exactly one complete JSON value.

Let's confirm that.


In [45]:
many_values = '''
{"id": 1}
{"id": 2}
[10, 20, 30]
"finished"
'''

try:
    json.loads(many_values)
except json.JSONDecodeError as exc:
    print(type(exc).__name__)
    print(exc)


JSONDecodeError
Extra data: line 3 column 1 (char 11)


For this situation, `JSONDecoder.raw_decode` is useful.

It decodes one value starting at a specific position and returns:

```python
(value, end_index)
```

The ending index tells us where that JSON value stopped.


## Step 1 — Decode only the first value


In [46]:
decoder = json.JSONDecoder()

text = '{"a": 1} {"b": 2}'

value, end = decoder.raw_decode(text)

print("value:", value)
print("end:", end)
print("remaining text:", repr(text[end:]))


value: {'a': 1}
end: 8
remaining text: ' {"b": 2}'


The decoder stopped after the first JSON value.

Now we can repeat the process.


## Step 2 — Write a loop

We need to:

1. skip whitespace,
2. call `raw_decode`,
3. store the value,
4. continue from the returned ending index.


In [47]:
def decode_many(text, decoder=None):
    decoder = decoder or json.JSONDecoder()

    values = []
    index = 0

    while index < len(text):

        # Skip whitespace before the next JSON value.
        while index < len(text) and text[index].isspace():
            index += 1

        if index >= len(text):
            break

        value, end = decoder.raw_decode(text, index)

        values.append(value)
        index = end

    return values


## Step 3 — Test the helper


In [48]:
values = decode_many(many_values)
pprint(values)


[{'id': 1}, {'id': 2}, [10, 20, 30], 'finished']


In [49]:
assert values == [
    {"id": 1},
    {"id": 2},
    [10, 20, 30],
    "finished",
]


## Step 4 — Use a custom decoder with `raw_decode`

Because `decode_many` accepts a decoder object, it can reuse all our custom
policies.


In [50]:
custom_text = '''
{"location": {"_type": "point", "x": 1.25, "y": 2.5}}
{"location": {"_type": "point", "x": 3.75, "y": 4.5}}
'''

custom_decoder = json.JSONDecoder(
    object_hook=extended_registry_hook,
    parse_float=Decimal
)

values = decode_many(
    custom_text,
    custom_decoder
)

pprint(values)


[{'location': Point(x=Decimal('1.25'), y=Decimal('2.5'))},
 {'location': Point(x=Decimal('3.75'), y=Decimal('4.5'))}]


# Problem 11 — When Overriding `decode` Actually Makes Sense

So far we mostly avoided overriding `decode`.

Now let's look at a case where overriding it is appropriate.

Suppose every valid document in our application must have this top-level shape:

```json
{
    "schema_version": 1,
    "payload": ...
}
```

This rule applies to the **entire decoded document**, not to each individual
nested object.

That is a good candidate for a custom `decode` method.


## Step 1 — Build the base decoder behavior

We still want:

- `Decimal` for floating-point numbers,
- tagged object conversion.


In [51]:
class ApplicationDecoder(json.JSONDecoder):
    def __init__(self, *args, **kwargs):
        kwargs.setdefault("parse_float", Decimal)
        kwargs.setdefault("object_hook", extended_registry_hook)

        super().__init__(*args, **kwargs)


## Step 2 — Add whole-document validation

We call `super().decode(...)` first.

That lets the normal decoder do all parsing and object-hook processing.

Then we validate the final top-level object.


In [52]:
class EnvelopeDecoder(ApplicationDecoder):
    def decode(self, s, _w=json.decoder.WHITESPACE.match):
        obj = super().decode(s, _w)

        if not isinstance(obj, dict):
            raise ValueError(
                "Top-level JSON value must be an object"
            )

        required = {"schema_version", "payload"}

        missing = required - set(obj)

        if missing:
            raise ValueError(
                f"Missing required top-level keys: {sorted(missing)}"
            )

        if obj["schema_version"] != 1:
            raise ValueError(
                f"Unsupported schema version: "
                f"{obj['schema_version']!r}"
            )

        return obj


## Step 3 — Test a valid document


In [53]:
j = '''
{
    "schema_version": 1,

    "payload": {
        "amount": 19.95,

        "position": {
            "_type": "point",
            "x": 10.5,
            "y": 20.25
        }
    }
}
'''

result = json.loads(
    j,
    cls=EnvelopeDecoder
)

pprint(result)


{'payload': {'amount': Decimal('19.95'),
             'position': Point(x=Decimal('10.5'), y=Decimal('20.25'))},
 'schema_version': 1}


In [54]:
assert result["schema_version"] == 1
assert result["payload"]["amount"] == Decimal("19.95")
assert result["payload"]["position"] == Point(
    Decimal("10.5"),
    Decimal("20.25")
)


## Step 4 — Test an invalid document


In [55]:
bad = '''
{
    "schema_version": 99,
    "payload": {}
}
'''

try:
    json.loads(bad, cls=EnvelopeDecoder)
except ValueError as exc:
    print("Rejected:", exc)


Rejected: Unsupported schema version: 99


## Problem 11 — Final lesson

Overriding `decode` is especially useful for:

- whole-document validation,
- top-level envelope checks,
- schema version requirements,
- cross-field invariants that only make sense after the document has been decoded.

It is less attractive when the only goal is to transform certain nested
dictionaries.


# Problem 12 — Build a Versioned Tagged Type

Real formats evolve.

Suppose point version 1 looked like this:

```json
{
    "_type": "point",
    "_version": 1,
    "x": 10,
    "y": 20
}
```

Later, version 2 changes the representation:

```json
{
    "_type": "point",
    "_version": 2,
    "coordinates": [10, 20]
}
```

We want both to decode to the same Python `Point`.


## Step 1 — Decode version 1


In [56]:
def decode_point_v1(obj):
    expected = {
        "_type",
        "_version",
        "x",
        "y",
    }

    if set(obj) != expected:
        raise ValueError("Malformed point version 1")

    return Point(
        obj["x"],
        obj["y"]
    )


## Step 2 — Decode version 2


In [57]:
def decode_point_v2(obj):
    expected = {
        "_type",
        "_version",
        "coordinates",
    }

    if set(obj) != expected:
        raise ValueError("Malformed point version 2")

    coordinates = obj["coordinates"]

    if (
        not isinstance(coordinates, list)
        or len(coordinates) != 2
    ):
        raise ValueError(
            "Version 2 coordinates must contain exactly 2 values"
        )

    return Point(
        coordinates[0],
        coordinates[1]
    )


## Step 3 — Dispatch according to `_version`

Notice that we do not silently guess a version.

If the version is unknown or absent, the decoder fails.


In [58]:
def versioned_point_hook(obj):
    if obj.get("_type") != "point":
        return obj

    version = obj.get("_version")

    if version == 1:
        return decode_point_v1(obj)

    if version == 2:
        return decode_point_v2(obj)

    raise ValueError(
        f"Unsupported or missing point version: {version!r}"
    )


## Step 4 — Test both representations


In [59]:
v1_json = '''
{
    "_type": "point",
    "_version": 1,
    "x": 10,
    "y": 20
}
'''

v2_json = '''
{
    "_type": "point",
    "_version": 2,
    "coordinates": [10, 20]
}
'''

p1 = json.loads(
    v1_json,
    object_hook=versioned_point_hook
)

p2 = json.loads(
    v2_json,
    object_hook=versioned_point_hook
)

print(p1)
print(p2)

assert p1 == Point(10, 20)
assert p2 == Point(10, 20)


Point(x=10, y=20)
Point(x=10, y=20)


## Problem 12 — Final lesson

Explicit version fields are useful when a JSON format is expected to live for a
long time.

They allow a decoder to support old data deliberately instead of guessing from
shape alone.


# Problem 13 — Build an Immutable `User` Object with Validation

Let's create a more realistic domain object.

A tagged user has this shape:

```json
{
    "_type": "user",
    "id": 123,
    "name": "Ada",
    "email": "ada@example.com",
    "roles": ["admin", "author"]
}
```

Rules:

- `id` must be a positive integer,
- `bool` must not be accepted as an integer ID,
- `name` must be a non-empty string,
- `email` must contain one `@`,
- `roles` must be a non-empty list,
- role names must be non-empty strings,
- duplicate roles are forbidden.


## Step 1 — Define the domain object


In [60]:
@dataclass(frozen=True)
class User:
    id: int
    name: str
    email: str
    roles: tuple


## Step 2 — Validate the ID

A subtle Python detail:

```python
isinstance(True, int)
```

is `True`.

So checking only `isinstance(value, int)` is not enough if booleans must be
rejected.


In [61]:
print(isinstance(True, int))


True


We'll explicitly reject booleans.


## Step 3 — Write the user decoder


In [62]:
def decode_user(obj):
    expected = {
        "_type",
        "id",
        "name",
        "email",
        "roles",
    }

    if set(obj) != expected:
        raise ValueError(
            f"Malformed user keys: {set(obj)}"
        )

    user_id = obj["id"]
    name = obj["name"]
    email = obj["email"]
    roles = obj["roles"]

    if (
        isinstance(user_id, bool)
        or not isinstance(user_id, int)
        or user_id <= 0
    ):
        raise ValueError(
            "User ID must be a positive integer"
        )

    if (
        not isinstance(name, str)
        or not name.strip()
    ):
        raise ValueError(
            "User name must be a non-empty string"
        )

    if (
        not isinstance(email, str)
        or email.count("@") != 1
    ):
        raise ValueError(
            "Invalid email for this exercise"
        )

    if (
        not isinstance(roles, list)
        or not roles
    ):
        raise ValueError(
            "Roles must be a non-empty list"
        )

    if not all(
        isinstance(role, str) and role
        for role in roles
    ):
        raise ValueError(
            "Each role must be a non-empty string"
        )

    if len(set(roles)) != len(roles):
        raise ValueError(
            "Duplicate roles are not allowed"
        )

    return User(
        id=user_id,
        name=name.strip(),
        email=email,
        roles=tuple(roles),
    )


## Step 4 — Add it to a hook


In [63]:
DOMAIN_DECODERS = {
    **EXTENDED_TYPE_DECODERS,
    "user": decode_user,
}


def domain_hook(obj):
    tag = obj.get("_type")

    if tag is None:
        return obj

    if tag not in DOMAIN_DECODERS:
        raise ValueError(
            f"Unknown domain type: {tag!r}"
        )

    return DOMAIN_DECODERS[tag](obj)


## Step 5 — Decode a valid user


In [64]:
j = '''
{
    "_type": "user",
    "id": 123,
    "name": "  Ada  ",
    "email": "ada@example.com",
    "roles": ["admin", "author"]
}
'''

user = json.loads(
    j,
    object_hook=domain_hook
)

print(user)


User(id=123, name='Ada', email='ada@example.com', roles=('admin', 'author'))


In [65]:
assert user == User(
    id=123,
    name="Ada",
    email="ada@example.com",
    roles=("admin", "author")
)


## Step 6 — Test several invalid users

A robust decoder needs negative tests too.


In [66]:
bad_users = [
    # bool should not count as an integer ID
    '''
    {
        "_type": "user",
        "id": true,
        "name": "Ada",
        "email": "ada@example.com",
        "roles": ["admin"]
    }
    ''',

    # blank name
    '''
    {
        "_type": "user",
        "id": 1,
        "name": "   ",
        "email": "ada@example.com",
        "roles": ["admin"]
    }
    ''',

    # duplicate role
    '''
    {
        "_type": "user",
        "id": 1,
        "name": "Ada",
        "email": "ada@example.com",
        "roles": ["admin", "admin"]
    }
    ''',
]

for text in bad_users:
    try:
        json.loads(
            text,
            object_hook=domain_hook
        )
    except ValueError as exc:
        print("Rejected:", exc)


Rejected: User ID must be a positive integer
Rejected: User name must be a non-empty string
Rejected: Duplicate roles are not allowed


# Problem 14 — JSON Lines with Error Reporting

JSON Lines (`.jsonl`) stores one JSON value per line.

Example:

```text
{"id": 1, "value": 10.5}
{"id": 2, "value": 20.5}
{"id": 3, "value":
{"id": 4, "value": 40.5}
```

One line is broken.

For batch processing, we may not want the entire import to stop because of one
bad record.

Let's build a helper that:

- decodes every non-empty line,
- stores valid values,
- stores errors with their line numbers.


## Step 1 — Decode one line at a time

Unlike concatenated JSON parsing with `raw_decode`, JSON Lines already tells us
where record boundaries are: the newline.


In [67]:
jsonl_text = '''
{"id": 1, "value": 10.5}
{"id": 2, "value": 20.5}
{"id": 3, "value":
{"id": 4, "value": 40.5}
'''


## Step 2 — Write the processor


In [68]:
def decode_json_lines(text, decoder=None):
    decoder = decoder or json.JSONDecoder()

    values = []
    errors = []

    for line_number, line in enumerate(
        text.splitlines(),
        start=1
    ):
        if not line.strip():
            continue

        try:
            value = decoder.decode(line)
        except Exception as exc:
            errors.append({
                "line": line_number,
                "text": line,
                "error_type": type(exc).__name__,
                "error": str(exc),
            })
        else:
            values.append(value)

    return values, errors


## Step 3 — Use a decoder with `Decimal`


In [69]:
jsonl_decoder = json.JSONDecoder(
    parse_float=Decimal
)

values, errors = decode_json_lines(
    jsonl_text,
    jsonl_decoder
)

print("VALUES")
pprint(values)

print("\nERRORS")
pprint(errors)


VALUES
[{'id': 1, 'value': Decimal('10.5')},
 {'id': 2, 'value': Decimal('20.5')},
 {'id': 4, 'value': Decimal('40.5')}]

ERRORS
[{'error': 'Expecting value: line 1 column 19 (char 18)',
  'error_type': 'JSONDecodeError',
  'line': 4,
  'text': '{"id": 3, "value":'}]


In [70]:
assert len(values) == 3
assert len(errors) == 1
assert values[0]["value"] == Decimal("10.5")


# Problem 15 — Produce Better `JSONDecodeError` Messages

Syntax errors already contain useful metadata.

`JSONDecodeError` provides:

- `msg`
- `pos`
- `lineno`
- `colno`

Let's build a small helper that displays the bad line and points to the
approximate error location.


## Step 1 — Inspect a normal `JSONDecodeError`


In [71]:
broken_json = '''
{
    "a": 1,
    "b": [10, 20,]
}
'''

try:
    json.loads(broken_json)
except json.JSONDecodeError as exc:
    print("message:", exc.msg)
    print("position:", exc.pos)
    print("line:", exc.lineno)
    print("column:", exc.colno)


message: Illegal trailing comma before end of array
position: 31
line: 4
column: 17


## Step 2 — Build a friendly formatter


In [72]:
def explain_json_error(text):
    try:
        return json.loads(text)

    except json.JSONDecodeError as exc:
        lines = text.splitlines()

        if 1 <= exc.lineno <= len(lines):
            line = lines[exc.lineno - 1]
        else:
            line = ""

        pointer = (
            " " * max(exc.colno - 1, 0)
            + "^"
        )

        return (
            f"{exc.msg}\n"
            f"line={exc.lineno}, "
            f"column={exc.colno}, "
            f"position={exc.pos}\n"
            f"{line}\n"
            f"{pointer}"
        )


## Step 3 — Test the formatter


In [73]:
print(explain_json_error(broken_json))


Illegal trailing comma before end of array
line=4, column=17, position=31
    "b": [10, 20,]
                ^


# Capstone Problem — Build a Strict Application Protocol Decoder

We will now combine several ideas into one decoder.

Our fictional protocol requires:

### Numeric rules

- integers must stay between `-1_000_000` and `1_000_000`,
- floating-point literals must become `Decimal`,
- `NaN` and infinities are forbidden.

### Object rules

- duplicate keys are forbidden,
- known tagged types may be reconstructed,
- unknown tagged types are errors.

### Document rules

The top-level object must contain:

- `schema_version`,
- `event_id`,
- `created_at`,
- `actor`,
- `payload`.

The `schema_version` must be `2`.

This problem is intentionally more involved, so we will assemble it in layers.


## Capstone Step 1 — Decide where each rule belongs

This is one of the most important design decisions.

| Rule | Best extension point |
|---|---|
| Decimal floats | `parse_float` |
| Bounded integers | `parse_int` |
| Reject NaN/Infinity | `parse_constant` |
| Duplicate keys | `object_pairs_hook` |
| Tagged object conversion | combined into `object_pairs_hook` |
| Top-level schema validation | overridden `decode` |


## Capstone Step 2 — Build the pair hook

Because `object_pairs_hook` takes priority over `object_hook`, we will combine:

1. duplicate-key checking,
2. tagged object conversion,

in one function.


In [74]:
PROTOCOL_DECODERS = {
    **DOMAIN_DECODERS,
}


def protocol_pairs_hook(pairs):
    obj = {}

    for key, value in pairs:
        if key in obj:
            raise ValueError(
                f"Duplicate JSON key: {key!r}"
            )

        obj[key] = value

    tag = obj.get("_type")

    if tag is None:
        return obj

    if tag not in PROTOCOL_DECODERS:
        raise ValueError(
            f"Unknown protocol type: {tag!r}"
        )

    return PROTOCOL_DECODERS[tag](obj)


## Capstone Step 3 — Build the decoder configuration


In [75]:
class ProtocolDecoder(json.JSONDecoder):
    def __init__(self, *args, **kwargs):
        kwargs["parse_int"] = bounded_int
        kwargs["parse_float"] = Decimal
        kwargs["parse_constant"] = reject_constant
        kwargs["object_pairs_hook"] = protocol_pairs_hook

        # If the caller supplied object_hook, remove it because
        # object_pairs_hook is the policy we want for this protocol.
        kwargs.pop("object_hook", None)

        super().__init__(*args, **kwargs)


At this point we have strict parsing and object reconstruction.

But we still have not checked the required top-level document shape.


## Capstone Step 4 — Add whole-document validation


In [76]:
class ProtocolDecoder(ProtocolDecoder):
    def decode(self, s, _w=json.decoder.WHITESPACE.match):
        obj = super().decode(s, _w)

        if not isinstance(obj, dict):
            raise ValueError(
                "Protocol document must be a JSON object"
            )

        required = {
            "schema_version",
            "event_id",
            "created_at",
            "actor",
            "payload",
        }

        missing = required - set(obj)

        if missing:
            raise ValueError(
                f"Missing required fields: {sorted(missing)}"
            )

        if obj["schema_version"] != 2:
            raise ValueError(
                f"Unsupported schema version: "
                f"{obj['schema_version']!r}"
            )

        if not isinstance(obj["event_id"], UUID):
            raise TypeError(
                "event_id must decode to UUID"
            )

        if not isinstance(obj["created_at"], datetime):
            raise TypeError(
                "created_at must decode to datetime"
            )

        if not isinstance(obj["actor"], User):
            raise TypeError(
                "actor must decode to User"
            )

        return obj


## Capstone Step 5 — Create a valid protocol document


In [77]:
protocol_json = '''
{
    "schema_version": 2,

    "event_id": {
        "_type": "uuid",
        "value": "550e8400-e29b-41d4-a716-446655440000"
    },

    "created_at": {
        "_type": "datetime",
        "value": "2026-08-07T14:30:00+00:00"
    },

    "actor": {
        "_type": "user",
        "id": 42,
        "name": "Ada",
        "email": "ada@example.com",
        "roles": ["admin", "author"]
    },

    "payload": {
        "amount": 125.75,

        "location": {
            "_type": "point",
            "x": 23.3219,
            "y": 42.6977
        },

        "labels": {
            "_type": "set",
            "items": ["priority", "audited"]
        }
    }
}
'''


## Capstone Step 6 — Decode it


In [78]:
protocol = json.loads(
    protocol_json,
    cls=ProtocolDecoder
)

pprint(protocol)


{'actor': User(id=42,
               name='Ada',
               email='ada@example.com',
               roles=('admin', 'author')),
 'created_at': datetime.datetime(2026, 8, 7, 14, 30, tzinfo=datetime.timezone.utc),
 'event_id': UUID('550e8400-e29b-41d4-a716-446655440000'),
 'payload': {'amount': Decimal('125.75'),
             'labels': {'priority', 'audited'},
             'location': Point(x=Decimal('23.3219'), y=Decimal('42.6977'))},
 'schema_version': 2}


## Capstone Step 7 — Verify every important transformation


In [79]:
assert protocol["schema_version"] == 2

assert isinstance(
    protocol["event_id"],
    UUID
)

assert isinstance(
    protocol["created_at"],
    datetime
)

assert isinstance(
    protocol["actor"],
    User
)

assert protocol["payload"]["amount"] == Decimal("125.75")

assert protocol["payload"]["location"] == Point(
    Decimal("23.3219"),
    Decimal("42.6977")
)

assert protocol["payload"]["labels"] == {
    "priority",
    "audited",
}


## Capstone Step 8 — Negative test: duplicate key


In [80]:
bad_duplicate = '''
{
    "schema_version": 2,
    "schema_version": 3,

    "event_id": {
        "_type": "uuid",
        "value": "550e8400-e29b-41d4-a716-446655440000"
    },

    "created_at": {
        "_type": "datetime",
        "value": "2026-08-07T14:30:00+00:00"
    },

    "actor": {
        "_type": "user",
        "id": 42,
        "name": "Ada",
        "email": "ada@example.com",
        "roles": ["admin"]
    },

    "payload": {}
}
'''

try:
    json.loads(
        bad_duplicate,
        cls=ProtocolDecoder
    )
except Exception as exc:
    print(type(exc).__name__ + ":", exc)


ValueError: Duplicate JSON key: 'schema_version'


## Capstone Step 9 — Negative test: unknown type


In [81]:
bad_type = '''
{
    "schema_version": 2,

    "event_id": {
        "_type": "uuid",
        "value": "550e8400-e29b-41d4-a716-446655440000"
    },

    "created_at": {
        "_type": "datetime",
        "value": "2026-08-07T14:30:00+00:00"
    },

    "actor": {
        "_type": "user",
        "id": 42,
        "name": "Ada",
        "email": "ada@example.com",
        "roles": ["admin"]
    },

    "payload": {
        "mystery": {
            "_type": "does_not_exist",
            "value": 123
        }
    }
}
'''

try:
    json.loads(
        bad_type,
        cls=ProtocolDecoder
    )
except Exception as exc:
    print(type(exc).__name__ + ":", exc)


ValueError: Unknown protocol type: 'does_not_exist'


## Capstone Step 10 — Negative test: invalid numeric constant


In [82]:
bad_number = '''
{
    "schema_version": 2,

    "event_id": {
        "_type": "uuid",
        "value": "550e8400-e29b-41d4-a716-446655440000"
    },

    "created_at": {
        "_type": "datetime",
        "value": "2026-08-07T14:30:00+00:00"
    },

    "actor": {
        "_type": "user",
        "id": 42,
        "name": "Ada",
        "email": "ada@example.com",
        "roles": ["admin"]
    },

    "payload": {
        "score": NaN
    }
}
'''

try:
    json.loads(
        bad_number,
        cls=ProtocolDecoder
    )
except Exception as exc:
    print(type(exc).__name__ + ":", exc)


ValueError: Non-standard numeric constant is not allowed: NaN


# Additional Guided Exercises

The following problems are intentionally shorter, but they are still designed
to be solved in stages.

Try them before looking back at similar patterns earlier in the notebook.


## Exercise A — A Tagged `Decimal`

Design this JSON representation:

```json
{
    "_type": "decimal",
    "value": "123.456"
}
```

Questions to think through:

1. Why is the decimal stored as a JSON string rather than a JSON floating-point
   literal?
2. What exception should be raised for `"value": "abc"`?
3. Should extra keys be accepted?


In [83]:
def decode_decimal(obj):
    if set(obj) != {"_type", "value"}:
        raise ValueError("Malformed decimal object")

    try:
        return Decimal(obj["value"])
    except Exception as exc:
        raise ValueError(
            f"Invalid decimal value: {obj['value']!r}"
        ) from exc


decimal_json = '''
{
    "_type": "decimal",
    "value": "123.456"
}
'''

value = json.loads(
    decimal_json,
    object_hook=lambda obj:
        decode_decimal(obj)
        if obj.get("_type") == "decimal"
        else obj
)

print(value)
assert value == Decimal("123.456")


123.456


## Exercise B — Case-Insensitive Duplicate Keys

For this exercise, these should count as duplicates:

```json
{
    "Role": "user",
    "role": "admin"
}
```

The first spelling should be preserved if there is no duplicate.


In [84]:
def no_case_insensitive_duplicates(pairs):
    result = {}
    seen = set()

    for key, value in pairs:
        normalized = key.casefold()

        if normalized in seen:
            raise ValueError(
                f"Duplicate key ignoring case: {key!r}"
            )

        seen.add(normalized)
        result[key] = value

    return result


In [85]:
good = '{"Name": "Ada", "Role": "admin"}'

print(json.loads(
    good,
    object_pairs_hook=no_case_insensitive_duplicates
))

bad = '{"Role": "user", "role": "admin"}'

try:
    json.loads(
        bad,
        object_pairs_hook=no_case_insensitive_duplicates
    )
except ValueError as exc:
    print("Rejected:", exc)


{'Name': 'Ada', 'Role': 'admin'}
Rejected: Duplicate key ignoring case: 'role'


## Exercise C — Track Start and End Positions with `raw_decode`

Modify our earlier `decode_many` idea so that each result contains:

```python
{
    "value": ...,
    "start": ...,
    "end": ...
}
```


In [86]:
def decode_many_with_positions(text, decoder=None):
    decoder = decoder or json.JSONDecoder()

    results = []
    index = 0

    while index < len(text):

        while (
            index < len(text)
            and text[index].isspace()
        ):
            index += 1

        if index >= len(text):
            break

        start = index

        value, end = decoder.raw_decode(
            text,
            index
        )

        results.append({
            "value": value,
            "start": start,
            "end": end,
        })

        index = end

    return results


In [87]:
text = '  {"a": 1}   [2, 3]  "done"'

pprint(
    decode_many_with_positions(text)
)


[{'end': 10, 'start': 2, 'value': {'a': 1}},
 {'end': 19, 'start': 13, 'value': [2, 3]},
 {'end': 27, 'start': 21, 'value': 'done'}]


# Best-Practice Checklist

When designing a custom JSON decoder, ask these questions.

### 1. Am I transforming individual JSON objects?

Prefer:

```python
object_hook
```

### 2. Do duplicate object keys matter?

Prefer:

```python
object_pairs_hook
```

### 3. Do I need exact decimal conversion?

Prefer:

```python
parse_float=Decimal
```

### 4. Do I need custom integer limits or integer types?

Use:

```python
parse_int
```

### 5. Do I need to reject `NaN` or infinities?

Use:

```python
parse_constant
```

### 6. Do I need to validate the entire decoded document?

That is a reasonable use case for overriding:

```python
JSONDecoder.decode
```

### 7. Do I have several custom object types?

Use an explicit registry instead of dynamic imports or arbitrary object
construction.

### 8. Is the input untrusted?

Fail closed:

- reject unknown tags,
- reject malformed tagged objects,
- reject duplicate keys when they are dangerous,
- validate expected fields,
- avoid executing or importing anything named by the JSON document.


# Final Challenge Set

Try these without looking at the earlier solutions.

### Challenge 1

Create a tagged `date` type using `datetime.date.fromisoformat`.

### Challenge 2

Create a tagged `money` type with:

- a 3-letter uppercase currency code,
- a `Decimal` amount.

### Challenge 3

Create a tagged `matrix` type and reject:

- empty matrices,
- rows with different lengths,
- non-numeric cells.

### Challenge 4

Create a decoder that accepts point versions 1, 2, and 3 and converts all of
them to the same `Point` class.

### Challenge 5

Create a strict decoder that rejects integer tokens containing more than
50 digits before converting them to `int`.

### Challenge 6

Create a JSON Lines loader that returns separate collections for:

- valid records,
- syntax errors,
- semantic validation errors.

### Challenge 7

Create a top-level envelope decoder where:

- `schema_version` must be 3,
- `payload` must be a dictionary,
- `created_at` must already have decoded to `datetime`.

### Challenge 8

Create a tagged expression tree supporting:

- numbers,
- addition,
- subtraction,
- multiplication,
- division.

Decode it into dataclasses and then evaluate it.

### Challenge 9

Add an explicit `_version` field to the `user` type and support two historical
representations.

### Challenge 10

Build tests proving that malformed tagged data never silently falls back to an
ordinary dictionary.


In [88]:
# Your workspace for the final challenge set.
#
# Add your own implementations below.
